# 🌧️ 実データで「係数の逆転」を体験する
## ― 多重共線性の罠：最高気温と最低気温を使った重回帰分析 ―

**データ出典：** FiveThirtyEight "29 Case Studies in Real Data Journalism" (CC BY 4.0)  
米国10都市・365日分の気象観測実測値

---

### この課題で起きること

| ステップ | 何をするか | 何が起きるか |
|--------|-----------|------------|
| 単回帰（最高気温のみ） | 「気温が高い日は雨が少ない」→ 係数は**負** | 直感と一致 ✅ |
| 単回帰（最低気温のみ） | 「気温が高い日は雨が少ない」→ 係数は**負** | 直感と一致 ✅ |
| **重回帰（両方投入）** | 最低気温の係数が突然**正**に逆転！ | ❓❓❓ |

**なぜこんなことが起きるのか？** ― それが今日の課題のテーマです。

---

### シナリオ

あなたは気象データを分析するデータサイエンティストです。  
「最高気温・最低気温から翌日の降水量を予測するモデルを作ってほしい」と依頼されました。  
さっそく重回帰分析をかけると…予想外の結果が待っていました。


## 0. ライブラリのインポート

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.graphics.gofplots import ProbPlot

from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

import matplotlib
#for font in ['IPAexGothic', 'Noto Sans CJK JP', 'Hiragino Sans']:
#    try: matplotlib.rc('font', family=font); break
#    except: pass
#plt.rcParams['axes.unicode_minus'] = False
!pip install japanize-matplotlib
import japanize_matplotlib

print('準備完了 ✅')


---
## 1. データ取得と前処理

FiveThirtyEight が公開している米国10都市の気象観測実測値を使います。

| 変数名 | 意味 | 単位 |
|--------|------|------|
| `date` | 日付 | — |
| `actual_max_temp` | 日最高気温 | °F |
| `actual_min_temp` | 日最低気温 | °F |
| `actual_mean_temp` | 日平均気温 | °F |
| `actual_precipitation` | 日降水量 | インチ |

対象都市：シアトル（KSEA）、ヒューストン（KHOU）ほか全10都市


In [ ]:
BASE = 'https://raw.githubusercontent.com/fivethirtyeight/data/master/us-weather-history/'
CITIES = {
    'KCLT': 'Charlotte, NC',   'KCQT': 'Los Angeles, CA',
    'KHOU': 'Houston, TX',     'KIND': 'Indianapolis, IN',
    'KJAX': 'Jacksonville, FL','KMDW': 'Chicago, IL',
    'KNYC': 'New York, NY',    'KPHL': 'Philadelphia, PA',
    'KPHX': 'Phoenix, AZ',     'KSEA': 'Seattle, WA',
}

dfs = []
for code_, name in CITIES.items():
    df_c = pd.read_csv(f'{BASE}{code_}.csv')
    df_c['city_code'] = code_
    df_c['city_name'] = name
    dfs.append(df_c)

df = pd.concat(dfs, ignore_index=True)
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month

df = df.dropna(subset=['actual_max_temp', 'actual_min_temp', 'actual_precipitation'])

print(f'全データ: {len(df):,} 行 × {df.shape[1]} 列')
print(f'都市数: {df["city_code"].nunique()}')
print(f'期間: {df["date"].min().date()} 〜 {df["date"].max().date()}')
print()
df[['actual_max_temp', 'actual_min_temp', 'actual_mean_temp', 'actual_precipitation']].describe().round(2)


---
## 2. まず「見る」― 探索的データ分析（EDA）

### ❓ 問題 2-1
以下のセルを実行して、散布図と相関行列を見てください。  
**最高気温と最低気温の相関係数はどれくらいですか？** 数値を読み取ってください。


In [ ]:
# シアトルを代表例として可視化
seattle = df[df['city_code'] == 'KSEA'].copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 最高気温 vs 降水量
axes[0].scatter(seattle['actual_max_temp'], seattle['actual_precipitation'],
               alpha=0.4, s=20, color='#2980b9', edgecolors='white', lw=0.3)
axes[0].set(xlabel='最高気温 (°F)', ylabel='降水量 (インチ)', title='最高気温 vs 降水量')

# 最低気温 vs 降水量
axes[1].scatter(seattle['actual_min_temp'], seattle['actual_precipitation'],
               alpha=0.4, s=20, color='#e74c3c', edgecolors='white', lw=0.3)
axes[1].set(xlabel='最低気温 (°F)', ylabel='降水量 (インチ)', title='最低気温 vs 降水量')

# 最高気温 vs 最低気温
axes[2].scatter(seattle['actual_max_temp'], seattle['actual_min_temp'],
               alpha=0.4, s=20, color='#27ae60', edgecolors='white', lw=0.3)
r = seattle['actual_max_temp'].corr(seattle['actual_min_temp'])
axes[2].set(xlabel='最高気温 (°F)', ylabel='最低気温 (°F)',
           title=f'最高気温 vs 最低気温  (r={r:.3f})')

plt.suptitle('シアトル（KSEA）2014年 気象データ', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'最高気温 と 最低気温 の相関係数 r = {r:.4f}')
print()
print('各変数と降水量の相関:')
print(seattle[['actual_max_temp','actual_min_temp','actual_precipitation']].corr().round(4))


In [ ]:
# 最高気温
plt.scatter((seattle['actual_max_temp']+seattle['actual_min_temp'])/2, seattle['actual_max_temp']-seattle['actual_min_temp'],
               alpha=0.4, s=20, color='#27ae60', edgecolors='white', lw=0.3)

In [ ]:
# 全10都市の 最高↔最低 相関係数を比較
print('都市別：最高気温 ↔ 最低気温 の相関係数')
print()
for code_, grp in df.groupby('city_code'):
    r = grp['actual_max_temp'].corr(grp['actual_min_temp'])
    bar = '█' * int(r * 30)
    print(f'{CITIES[code_]:20s}  r = {r:.4f}  {bar}')


---
## 3. 単回帰分析 ― まず1変数ずつ

重回帰の前に、説明変数を1つずつ使った単回帰で係数の**符号と大きさを確認**します。

### ❓ 問題 3-1
以下のセルを実行して結果を確認してください。  
「最高気温が 1°F 上がると、降水量はどれだけ変わりますか？」  
「最低気温が 1°F 上がると、降水量はどれだけ変わりますか？」  
係数の**符号**（正か負か）に注目してください。


In [ ]:
seattle = df[df['city_code'] == 'KSEA'].copy()
y = seattle['actual_precipitation']

# ── 単回帰：最高気温のみ ──────────────────────────────
X_max = sm.add_constant(seattle['actual_max_temp'])
m_max = sm.OLS(y, X_max).fit()
print('=' * 55)
print('  単回帰モデル A：降水量 ～ 最高気温')
print('=' * 55)
print(m_max.summary())


In [ ]:
# ── 単回帰：最低気温のみ ──────────────────────────────
X_min = sm.add_constant(seattle['actual_min_temp'])
m_min = sm.OLS(y, X_min).fit()
print('=' * 55)
print('  単回帰モデル B：降水量 ～ 最低気温')
print('=' * 55)
print(m_min.summary())


In [ ]:
# 単回帰の回帰直線を可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, feat, model, color, title in zip(
    axes,
    ['actual_max_temp', 'actual_min_temp'],
    [m_max, m_min],
    ['#2980b9', '#e74c3c'],
    ['モデル A：降水量 ～ 最高気温', 'モデル B：降水量 ～ 最低気温']
):
    ax.scatter(seattle[feat], y, alpha=0.3, s=15, color=color,
               edgecolors='white', lw=0.3)
    xs = np.linspace(seattle[feat].min(), seattle[feat].max(), 100)
    ax.plot(xs, model.params[0] + model.params[1] * xs,
            'k-', lw=2, alpha=0.8)
    coef = model.params[1]
    ax.set(xlabel=feat.replace('actual_','').replace('_',' '),
           ylabel='降水量 (インチ)',
           title=f'{title}\nβ = {coef:.5f}  ({'負 (↓)' if coef < 0 else '正 (↑)'})')

plt.suptitle('シアトル：単回帰の結果', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('【単回帰の係数まとめ】')
print(f'  モデル A  β(最高気温) = {m_max.params["actual_max_temp"]:+.5f}  '
      f'  → 気温が高い日ほど {"雨が少ない ✅" if m_max.params["actual_max_temp"] < 0 else "雨が多い"}')
print(f'  モデル B  β(最低気温) = {m_min.params["actual_min_temp"]:+.5f}  '
      f'  → 気温が高い日ほど {"雨が少ない ✅" if m_min.params["actual_min_temp"] < 0 else "雨が多い"}')


---
## 4. 重回帰分析 ― 両方まとめて投入

### ❓ 問題 4-1（予想）
最高気温も最低気温も「高いほど雨が少ない（係数は負）」という単回帰結果でした。  
**両方を同時にモデルに入れると、係数はどうなると思いますか？**  
（実行前に予想を書いてみてください）

> あなたの予想：
> - β(最高気温) の符号：＿＿＿
> - β(最低気温) の符号：＿＿＿


In [ ]:
# ── 重回帰：最高気温 + 最低気温 ───────────────────────
X_both = sm.add_constant(seattle[['actual_max_temp', 'actual_min_temp']])
m_both = sm.OLS(y, X_both).fit()

print('=' * 60)
print('  重回帰モデル C：降水量 ～ 最高気温 ＋ 最低気温')
print('=' * 60)
print(m_both.summary())


In [ ]:
# ── 係数の逆転を視覚的に示す ─────────────────────────
labels = ['最高気温', '最低気温']
b_single = [m_max.params['actual_max_temp'], m_min.params['actual_min_temp']]
b_multi  = [m_both.params['actual_max_temp'], m_both.params['actual_min_temp']]

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(2)
w = 0.35

bars_s = ax.bar(x - w/2, b_single, w, label='単回帰', color='#95a5a6', alpha=0.85, edgecolor='white')
bars_m = ax.bar(x + w/2, b_multi,  w, label='重回帰', color=['#2980b9','#e74c3c'], alpha=0.85, edgecolor='white')

ax.axhline(0, color='black', lw=1)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('回帰係数')
ax.set_title('単回帰 vs 重回帰：係数の比較\n（シアトル・降水量モデル）',
             fontsize=13, fontweight='bold')
ax.legend()

# 逆転した変数に矢印アノテーション
for i, (bs, bm) in enumerate(zip(b_single, b_multi)):
    if (bs > 0) != (bm > 0):
        ax.annotate('← 符号逆転！',
                   xy=(i + w/2, bm),
                   xytext=(i + w/2 + 0.15, bm + (0.003 if bm > 0 else -0.003)),
                   fontsize=10, color='red', fontweight='bold',
                   arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.show()

print('【係数の比較】')
print(f'{"変数":12s}  {"単回帰":>10}  {"重回帰":>10}  {"逆転?":>8}')
print('-' * 45)
for lbl, bs, bm in zip(labels, b_single, b_multi):
    flip = '★ 逆転！' if (bs > 0) != (bm > 0) else ''
    print(f'{lbl:12s}  {bs:>+10.5f}  {bm:>+10.5f}  {flip}')


---
## 5. なぜ逆転するのか ― 多重共線性の仕組みを解剖する

### 5-1. VIF（分散膨張因子）で定量化

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

$R^2_j$：変数 $j$ を**他の変数だけ**で回帰したときの決定係数。  
最高気温と最低気温が高相関なら $R^2_j$ が大きくなり → VIF が大きくなる。


In [ ]:
vif_vals = [variance_inflation_factor(X_both.values, i) for i in range(X_both.shape[1])]
vif_df = pd.DataFrame({
    '変数': ['定数項', '最高気温', '最低気温'],
    'VIF':  vif_vals
})

print('VIF（分散膨張因子）:')
print(vif_df.to_string(index=False))
print()
print('参考: VIF < 5 = 問題なし, VIF 5〜10 = 要注意, VIF > 10 = 多重共線性あり')

r = seattle['actual_max_temp'].corr(seattle['actual_min_temp'])
vif_theory = 1 / (1 - r**2)
print(f'\n理論値：r={r:.4f} → VIF = 1/(1-r²) = {vif_theory:.2f}')


### 5-2. 係数が逆転する幾何学的な理由

$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$

$X^\top X$ が**ほぼ特異行列**（最高気温と最低気温がほぼ同じ情報）になると、  
その逆行列の要素が非常に大きくなり、**係数の推定値が不安定**になる。

数値でも確認してみよう。


In [ ]:
# X^T X の条件数と固有値を確認
X_vals = X_both.values
XtX = X_vals.T @ X_vals
eigvals = np.linalg.eigvals(XtX)
cond_num = np.sqrt(eigvals.max() / eigvals.min())

print(f'条件数 κ = √(λmax/λmin) = {cond_num:.1f}')
print(f'（目安: 30以上で多重共線性を疑う、1000以上で深刻）')
print()
print('固有値:', np.sort(eigvals)[::-1].round(2))


### 5-3. 係数逆転のシミュレーション ― 相関係数を変えると何が起きるか

In [ ]:
# 相関が変わると係数と標準誤差がどう変わるかをシミュレーション
np.random.seed(42)
n = 365
rho_list = np.linspace(0.0, 0.98, 40)
coef_max_list, coef_min_list = [], []
se_max_list, se_min_list = [], []

true_beta_max = -0.012  # 真の最高気温係数（シアトルの重回帰値に近い）
true_beta_min = +0.015  # 真の最低気温係数

for rho in rho_list:
    # rho相関の2変数を生成
    tmax = np.random.normal(64, 13, n)
    tmin = rho * tmax + np.sqrt(1 - rho**2) * np.random.normal(48, 9, n)
    precip = 0.1 + true_beta_max * tmax + true_beta_min * tmin + np.random.normal(0, 0.2, n)
    precip = np.clip(precip, 0, None)

    Xs = sm.add_constant(np.column_stack([tmax, tmin]))
    try:
        res = sm.OLS(precip, Xs).fit()
        coef_max_list.append(res.params[1])
        coef_min_list.append(res.params[2])
        se_max_list.append(res.bse[1])
        se_min_list.append(res.bse[2])
    except:
        coef_max_list.append(np.nan); coef_min_list.append(np.nan)
        se_max_list.append(np.nan); se_min_list.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 係数の変化
axes[0].plot(rho_list, coef_max_list, '-o', ms=4, color='#2980b9',
             label='β(最高気温)', linewidth=1.5)
axes[0].plot(rho_list, coef_min_list, '-o', ms=4, color='#e74c3c',
             label='β(最低気温)', linewidth=1.5)
axes[0].axhline(0, color='black', lw=0.8, linestyle='--', alpha=0.5)
axes[0].axhline(true_beta_max, color='#2980b9', lw=1, linestyle=':', alpha=0.7, label='真の値')
axes[0].axhline(true_beta_min, color='#e74c3c', lw=1, linestyle=':', alpha=0.7)
axes[0].axvline(seattle['actual_max_temp'].corr(seattle['actual_min_temp']),
               color='green', lw=1.5, linestyle='--', label='シアトルの実際の r')
axes[0].set(xlabel='最高・最低気温の相関係数 r', ylabel='係数の推定値',
           title='相関係数と係数の推定値')
axes[0].legend(fontsize=9)

# 標準誤差の変化
axes[1].plot(rho_list, se_max_list, '-o', ms=4, color='#2980b9',
             label='SE(最高気温)', linewidth=1.5)
axes[1].plot(rho_list, se_min_list, '-o', ms=4, color='#e74c3c',
             label='SE(最低気温)', linewidth=1.5)
axes[1].axvline(seattle['actual_max_temp'].corr(seattle['actual_min_temp']),
               color='green', lw=1.5, linestyle='--', label='シアトルの実際の r')
axes[1].set(xlabel='最高・最低気温の相関係数 r', ylabel='標準誤差 (SE)',
           title='相関係数と係数の標準誤差（不確実性）')
axes[1].legend(fontsize=9)

plt.suptitle('多重共線性の影響：相関係数を変えたシミュレーション (n=365)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 r が 0.8 を超えたあたりから SE が急激に膨らむことを確認しよう')


---
## 6. 全10都市で係数逆転を確認

### ❓ 問題 6-1
以下のセルを実行して、全都市の結果を比較してください。  
- 係数逆転が起きている都市はどこですか？
- 逆転が起きやすい都市と起きにくい都市、何が違いますか？


In [ ]:
results = []
for code_, grp in df.groupby('city_code'):
    grp = grp.dropna(subset=['actual_max_temp','actual_min_temp','actual_precipitation'])
    y_ = grp['actual_precipitation']

    m_mx = sm.OLS(y_, sm.add_constant(grp['actual_max_temp'])).fit()
    m_mn = sm.OLS(y_, sm.add_constant(grp['actual_min_temp'])).fit()
    X_ = sm.add_constant(grp[['actual_max_temp','actual_min_temp']])
    m_bt = sm.OLS(y_, X_).fit()

    r_ = grp['actual_max_temp'].corr(grp['actual_min_temp'])
    vif_ = variance_inflation_factor(X_.values, 1)

    flip_max = (m_mx.params['actual_max_temp'] > 0) != (m_bt.params['actual_max_temp'] > 0)
    flip_min = (m_mn.params['actual_min_temp'] > 0) != (m_bt.params['actual_min_temp'] > 0)

    results.append({
        '都市': CITIES[code_],
        'r(max,min)': round(r_, 3),
        'VIF': round(vif_, 1),
        '単回帰β(max)': round(m_mx.params['actual_max_temp'], 5),
        '単回帰β(min)': round(m_mn.params['actual_min_temp'], 5),
        '重回帰β(max)': round(m_bt.params['actual_max_temp'], 5),
        '重回帰β(min)': round(m_bt.params['actual_min_temp'], 5),
        '最高逆転': '★' if flip_max else '',
        '最低逆転': '★' if flip_min else '',
    })

res_df = pd.DataFrame(results).sort_values('r(max,min)', ascending=False)
print(res_df.to_string(index=False))


In [ ]:
# 都市別の係数比較 バーチャート
fig, axes = plt.subplots(2, 5, figsize=(16, 7), sharey=False)
axes = axes.flatten()

for ax, (code_, grp) in zip(axes, df.groupby('city_code')):
    grp = grp.dropna(subset=['actual_max_temp','actual_min_temp','actual_precipitation'])
    y_ = grp['actual_precipitation']
    m_mx = sm.OLS(y_, sm.add_constant(grp['actual_max_temp'])).fit()
    m_mn = sm.OLS(y_, sm.add_constant(grp['actual_min_temp'])).fit()
    X_ = sm.add_constant(grp[['actual_max_temp','actual_min_temp']])
    m_bt = sm.OLS(y_, X_).fit()

    b_s = [m_mx.params['actual_max_temp'], m_mn.params['actual_min_temp']]
    b_m = [m_bt.params['actual_max_temp'], m_bt.params['actual_min_temp']]

    x = np.arange(2)
    ax.bar(x - 0.2, b_s, 0.38, color='#95a5a6', alpha=0.8, label='単回帰')
    colors = []
    for bs_, bm_ in zip(b_s, b_m):
        colors.append('#e74c3c' if (bs_ > 0) != (bm_ > 0) else '#3498db')
    ax.bar(x + 0.2, b_m, 0.38, color=colors, alpha=0.8, label='重回帰')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(['最高', '最低'], fontsize=9)
    r_ = grp['actual_max_temp'].corr(grp['actual_min_temp'])
    ax.set_title(f'{CITIES[code_].split(",")[0]}\nr={r_:.3f}', fontsize=9)

axes[0].legend(fontsize=8, loc='upper right')
plt.suptitle('全10都市：単回帰 vs 重回帰 係数比較（赤 = 逆転）',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## 7. 回帰診断 ― モデルの前提を検証する

多重共線性の問題だけでなく、OLS の他の仮定も確認しましょう。


In [ ]:
seattle = df[df['city_code'] == 'KSEA'].copy()
y = seattle['actual_precipitation']
X_both = sm.add_constant(seattle[['actual_max_temp', 'actual_min_temp']])
m_both = sm.OLS(y, X_both).fit()

resid   = m_both.resid
fitted  = m_both.fittedvalues
influence = OLSInfluence(m_both)
std_resid = influence.resid_studentized_internal

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 3, figure=fig)

# ① 残差 vs 予測値
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(fitted, resid, alpha=0.3, s=15, color='#2980b9')
ax1.axhline(0, color='red', lw=1.2, linestyle='--')
ax1.set(xlabel='予測値', ylabel='残差', title='① 残差 vs 予測値')

# ② Q-Q プロット
ax2 = fig.add_subplot(gs[0, 1])
ProbPlot(resid).qqplot(line='s', ax=ax2, alpha=0.3, markersize=3, color='#2980b9')
ax2.set_title('② Q-Q プロット')

# ③ Scale-Location
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.3, s=15, color='#27ae60')
ax3.set(xlabel='予測値', ylabel='√|標準化残差|', title='③ Scale-Location')

# ④ 残差のヒストグラム
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(resid, bins=40, color='#2980b9', edgecolor='white', alpha=0.85)
ax4.set(xlabel='残差', ylabel='件数', title='④ 残差のヒストグラム')

# ⑤ Cook's D
cooks_d = influence.cooks_distance[0]
ax5 = fig.add_subplot(gs[1, 1])
ax5.stem(np.arange(len(cooks_d)), cooks_d, markerfmt=',', linefmt='grey', basefmt='k-')
thresh = 4 / len(seattle)
ax5.axhline(thresh, color='red', lw=1.5, linestyle='--', label=f'4/n={thresh:.4f}')
ax5.set(xlabel='観測番号', ylabel="Cook's D", title="⑤ Cook's Distance")
ax5.legend(fontsize=8)

# ⑥ 月別の残差 boxplot（季節性の確認）
ax6 = fig.add_subplot(gs[1, 2])
seattle_with_resid = seattle.copy()
seattle_with_resid['resid'] = resid.values
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
resid_by_month = [seattle_with_resid[seattle_with_resid['month'] == m]['resid'].values
                  for m in range(1, 13)]
ax6.boxplot(resid_by_month, labels=month_labels, patch_artist=True,
           boxprops=dict(facecolor='#aed6f1'))
ax6.axhline(0, color='red', lw=1, linestyle='--')
ax6.set(xlabel='月', ylabel='残差', title='⑥ 月別の残差分布')

plt.suptitle('重回帰モデル C の回帰診断（シアトル）',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# 数値検定
from statsmodels.stats.stattools import durbin_watson
bp_lm, bp_p, _, _ = het_breuschpagan(resid, X_both)
dw = durbin_watson(resid)
print(f'Breusch-Pagan 検定（等分散）: LM={bp_lm:.2f}, p={bp_p:.4f}  →',
      '不均一分散あり' if bp_p < 0.05 else '均一分散')
print(f'Durbin-Watson（独立性）     : DW={dw:.3f}  →',
      '正の系列相関' if dw < 1.5 else '問題なし' if dw < 2.5 else '負の系列相関')
print(f"Cook's D > 4/n の観測: {(cooks_d > thresh).sum()} 件")


---
## 8. 対処法

多重共線性への対処には主に3つのアプローチがあります。

| 方法 | 内容 | 利点・欠点 |
|------|------|-----------|
| **変数削除** | 相関の高い変数の片方を除く | シンプルだが情報を捨てる |
| **新変数の作成** | 平均気温や気温差を特徴量に変換 | 解釈しやすい |
| **正則化** | Ridge / Lasso で係数を安定化 | 情報を保ちつつ安定化 |


In [ ]:
seattle = df[df['city_code'] == 'KSEA'].copy()
y = seattle['actual_precipitation'].values

# ── 方法1：変数削除（最高気温のみ残す）─────────────────
X_max_only = sm.add_constant(seattle['actual_max_temp'])
m_max_only  = sm.OLS(y, X_max_only).fit()

# ── 方法2：新変数（平均気温・気温差）────────────────────
seattle['avg_temp']   = (seattle['actual_max_temp'] + seattle['actual_min_temp']) / 2
seattle['temp_range'] = seattle['actual_max_temp'] - seattle['actual_min_temp']

X_eng = sm.add_constant(seattle[['avg_temp', 'temp_range']])
m_eng = sm.OLS(y, X_eng).fit()
r_eng = seattle['avg_temp'].corr(seattle['temp_range'])
vif_eng = variance_inflation_factor(X_eng.values, 1)

# ── 方法3：Ridge 回帰 ─────────────────────────────────
scaler   = StandardScaler()
X_raw    = seattle[['actual_max_temp', 'actual_min_temp']].values
X_sc     = scaler.fit_transform(X_raw)
alphas   = np.logspace(-3, 3, 200)
ridge_cv = RidgeCV(alphas=alphas, cv=10).fit(X_sc, y)
lasso_cv = LassoCV(alphas=alphas, cv=10, max_iter=10000, random_state=42).fit(X_sc, y)

# 比較表
print('=' * 70)
print(f'  {"モデル":<22} {"β(max/avg)":>12} {"β(min/range)":>13} {"VIF":>7} {"R²":>8}')
print('=' * 70)

X_both_ = sm.add_constant(seattle[['actual_max_temp','actual_min_temp']])
m_both_ = sm.OLS(y, X_both_).fit()
vif_b = variance_inflation_factor(X_both_.values, 1)
print(f'  {"重回帰（両方）":22}  {m_both_.params[1]:>+11.5f}  {m_both_.params[2]:>+12.5f}  {vif_b:>7.1f}  {m_both_.rsquared:>7.4f}')

vif_d = variance_inflation_factor(sm.add_constant(seattle['actual_max_temp'].values.reshape(-1,1)).reshape(-1,2) if False else np.column_stack([np.ones(len(seattle)), seattle['actual_max_temp']]), 1)
print(f'  {"方法1：最高気温のみ":22}  {m_max_only.params[1]:>+11.5f}  {"—":>13}  {"—":>7}  {m_max_only.rsquared:>7.4f}')
print(f'  {"方法2：平均＋気温差":22}  {m_eng.params[1]:>+11.5f}  {m_eng.params[2]:>+12.5f}  {vif_eng:>7.1f}  {m_eng.rsquared:>7.4f}')
print(f'  {"方法3：Ridge(標準化)":22}  {ridge_cv.coef_[0]:>+11.5f}  {ridge_cv.coef_[1]:>+12.5f}  {"—":>7}  {ridge_cv.score(X_sc,y):>7.4f}')
print(f'  {"方法3：Lasso(標準化)":22}  {lasso_cv.coef_[0]:>+11.5f}  {lasso_cv.coef_[1]:>+12.5f}  {"—":>7}  {lasso_cv.score(X_sc,y):>7.4f}')
print()
print(f'※ 方法2の相関 r(平均気温, 気温差) = {r_eng:.4f}  VIF = {vif_eng:.2f}')
print('✅ 方法2で VIF が大幅に改善（多重共線性が解消）')


In [ ]:
# 特徴量エンジニアリングの効果を可視化
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# 重回帰（問題あり）
ax = axes[0]
ax.scatter(m_both_.fittedvalues, m_both_.resid, alpha=0.3, s=15, color='#e74c3c')
ax.axhline(0, color='black', lw=1, linestyle='--')
ax.set(xlabel='予測値', ylabel='残差', title=f'重回帰（両方）\nR²={m_both_.rsquared:.4f}, VIF={vif_b:.1f}')

# 方法2（平均気温＋気温差）
ax = axes[1]
ax.scatter(m_eng.fittedvalues, m_eng.resid, alpha=0.3, s=15, color='#27ae60')
ax.axhline(0, color='black', lw=1, linestyle='--')
ax.set(xlabel='予測値', ylabel='残差', title=f'方法2（平均＋気温差）\nR²={m_eng.rsquared:.4f}, VIF={vif_eng:.2f}')

# 係数比較
ax = axes[2]
methods = ['重回帰\n（問題あり）', '方法2\n（解決策）', 'Ridge\n（標準化）']
b_maxlike = [m_both_.params[1], m_eng.params[1], ridge_cv.coef_[0]]
b_minlike = [m_both_.params[2], m_eng.params[2], ridge_cv.coef_[1]]
x = np.arange(3)
ax.bar(x - 0.2, b_maxlike, 0.38, color='#2980b9', alpha=0.85, label='β(最高/平均/max)')
ax.bar(x + 0.2, b_minlike, 0.38, color='#e74c3c', alpha=0.85, label='β(最低/気温差/min)')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=9)
ax.set_ylabel('係数')
ax.set_title('対処法ごとの係数比較')
ax.legend(fontsize=8)

plt.suptitle('対処法の効果比較', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 9. AIC / BIC によるモデル選択

### ❓ 問題 9-1
以下の5つのモデルを AIC / BIC で比較してください。  
「係数が逆転しているモデル C」と、対処法を施したモデルでは、どちらが良いと評価されますか？


In [ ]:
seattle = df[df['city_code'] == 'KSEA'].copy()
y = seattle['actual_precipitation']
seattle['avg_temp']   = (seattle['actual_max_temp'] + seattle['actual_min_temp']) / 2
seattle['temp_range'] = seattle['actual_max_temp'] - seattle['actual_min_temp']

model_specs = {
    'M1: 最高気温のみ':          ['actual_max_temp'],
    'M2: 最低気温のみ':          ['actual_min_temp'],
    'M3: 重回帰（両方）':        ['actual_max_temp', 'actual_min_temp'],
    'M4: 平均気温＋気温差':      ['avg_temp', 'temp_range'],
    'M5: 月ダミー追加':          ['avg_temp', 'temp_range'] + [f'month_{m}' for m in range(2,13)],
}

# 月ダミー変数を追加
for m in range(2, 13):
    seattle[f'month_{m}'] = (seattle['month'] == m).astype(float)

results = []
for name, cols in model_specs.items():
    try:
        m_ = sm.OLS(y, sm.add_constant(seattle[cols])).fit()
        results.append({
            'モデル': name, 'k': len(cols)+1,
            'R²': round(m_.rsquared, 4), 'Adj-R²': round(m_.rsquared_adj, 4),
            'AIC': round(m_.aic, 1), 'BIC': round(m_.bic, 1),
            'RMSE': round(np.sqrt(m_.mse_resid), 4),
        })
    except Exception as e:
        print(f'{name}: {e}')

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bar_colors = ['#3498db','#e67e22','#e74c3c','#27ae60','#9b59b6']
for ax, metric in zip(axes, ['AIC', 'BIC']):
    bars = ax.bar(res_df['モデル'], res_df[metric],
                  color=bar_colors, edgecolor='white', width=0.6)
    best = res_df[metric].idxmin()
    bars[best].set_edgecolor('gold'); bars[best].set_linewidth(3)
    ax.set_xticklabels(res_df['モデル'], rotation=20, ha='right', fontsize=8)
    ax.set(ylabel=metric, title=f'{metric}（金枠=最良）')
    for bar, val in zip(bars, res_df[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()-0.3,
                f'{val:.0f}', ha='center', va='top', fontsize=8, color='white')
plt.suptitle('AIC / BIC モデル比較（シアトル・降水量モデル）', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 10. まとめと考察

### 今回の発見

| 現象 | 原因 | 対処 |
|------|------|------|
| 単回帰では両方「負」→ 重回帰で一方が「正」に逆転 | 最高・最低気温の高相関（r≈0.86）による多重共線性 | 変数削除 / 特徴量エンジニアリング / 正則化 |
| 重回帰の R² が高いのに係数の p 値が大きい | VIF が高く SE が膨らむ | VIF で定量確認 |
| 係数が都市によって逆転したりしなかったりする | 都市によって気候パターン（r の大きさ）が異なる | データ依存性を理解 |

### ❓ 最終課題（考察）

1. **なぜシアトルで特に係数逆転が起きやすいのか？** シアトルの気候の特徴と結びつけて考察してください。

2. **「方法2（平均気温＋気温差）」はなぜ多重共線性を解消できるのか？** 数式で説明してください。  
   ヒント：$x_1 = T_{\max}$、$x_2 = T_{\min}$ とするとき、$\bar{T} = (x_1+x_2)/2$、$D = x_1 - x_2$ と置くと…

3. **AIC/BIC が最良と判定したモデルと、VIF が問題ないモデルは一致しましたか？** なぜそうなったか考察してください。

4. **全10都市の重回帰を同時に推定する「パネルデータ分析（固定効果モデル）」** では、多重共線性への影響はどう変わるか調べてみてください。
